# RIN — эксперимент 4: семантический вентиль `rel(B|q)` в `diffuse()`

Тот же самый golden-микромир, что в эксперименте 3 (coffee/caffeine/beverage + north dakota), и та же самая ячейка сценария 2 — но с `query_vectors=` в `diffuse()`.

**Гипотеза (из эксперимента 3):** без вентиля `beverage` (146.67) обгоняет `caffeine` (26.67) в 5.5 раза. С вентилем `caffeine`, как семантически близкий к query, должен выиграть.

In [1]:
import sys
from pathlib import Path

def find_repo_root(marker="requirements.txt"):
    path = Path.cwd()
    for candidate in [path, *path.parents]:
        if (candidate / marker).exists():
            return candidate
    raise FileNotFoundError(f"could not find repo root (looked for {marker})")

REPO_ROOT = find_repo_root()
SRC = REPO_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

In [2]:
%load_ext autoreload
%autoreload 2

## Тот же граф, что в эксперименте 3 (дублируем построение — микромир маленький, не жалко)

In [3]:
from embryonic.numberbatch import Numberbatch

NUMBERBATCH_FILE = REPO_ROOT / "source" / "numberbatch-en-19.08.txt.gz"
numberbatch = Numberbatch()
numberbatch.load(str(NUMBERBATCH_FILE))
print(f"{len(numberbatch.terms):,} терминов")

516,782 терминов


In [4]:
from memory.graph import MemoryGraph
from memory.node import Node
from memory.edge import Edge

NODE_SPECS = [
    ("coffee", "/c/en/coffee", "concept", 0.7),
    ("caffeine", "/c/en/caffeine", "concept", 0.7),
    ("beverage", "/c/en/beverage", "concept", 0.7),
    ("hot drink", "/c/en/hot_drink", "concept", 0.7),
    ("tea", "/c/en/tea", "concept", 0.7),
    ("juice", "/c/en/juice", "concept", 0.7),
    ("soda", "/c/en/soda", "concept", 0.7),
    ("north dakota", "/c/en/north_dakota", "concept", 0.7),
    ("south dakota", "/c/en/south_dakota", "concept", 0.7),
    ("north", "/c/en/north", "concept", 0.7),
]

EDGE_SPECS = [
    ("coffee", "caffeine", "HasProperty", 1.0),
    ("coffee", "beverage", "IsA", 1.0),
    ("coffee", "hot drink", "RelatedTo", 1.0),
    ("hot drink", "beverage", "IsA", 1.0),
    ("beverage", "tea", "RelatedTo", 1.0),
    ("beverage", "juice", "RelatedTo", 1.0),
    ("beverage", "soda", "RelatedTo", 1.0),
    ("north dakota", "south dakota", "RelatedTo", 1.0),
]

G = MemoryGraph()
content_to_id = {}
for i, (content, uri, kind, conf) in enumerate(NODE_SPECS):
    vector = numberbatch.vector_for(uri)
    assert vector is not None, f"no vector for {uri}"
    G.add_node(Node(id=i, content=content, embedding=vector, node_kind=kind,
                     confidence=conf, provenance=[uri]))
    content_to_id[content] = i

for src, dst, etype, weight in EDGE_SPECS:
    G.add_edge(Edge(src=content_to_id[src], dst=content_to_id[dst], type=etype, weight=weight))
    G.add_edge(Edge(src=content_to_id[dst], dst=content_to_id[src], type=etype, weight=weight))

print(f"{len(G.nodes)} узлов")

10 узлов


## Сценарий 2 повторно: без вентиля vs с вентилем

Тот же запрос, что в эксперименте 3 — "why coffee gives energy", caffeine не упоминается напрямую.

In [5]:
from embryonic.query import longest_match_terms, query_vectors
from memory.retrieval import nearest_nodes_by_embedding
from memory.diffuse import diffuse

QUERY_2 = "why coffee gives energy"
vecs_2 = query_vectors(QUERY_2, numberbatch)
print("query terms matched:", longest_match_terms(QUERY_2, numberbatch))

anchors_2 = set()
for v in vecs_2:
    anchors_2.update(nearest_nodes_by_embedding(G, v, top_k=3))
print("Якоря:", [G.nodes[a].content for a in anchors_2])

source_amplitudes_2 = {a: 100.0 for a in anchors_2}

query terms matched: ['/c/en/coffee', '/c/en/gives', '/c/en/energy']
Якоря: ['coffee', 'hot drink', 'tea']


In [6]:
# БЕЗ вентиля (query_vectors=None, дефолт) — воспроизводим эксперимент 3
lit_baseline = diffuse(G, source_amplitudes_2, decay=0.8, min_propagate_fraction=0.01, max_steps=20)

print("БЕЗ вентиля:")
for nid, amp in sorted(lit_baseline.items(), key=lambda kv: -kv[1]):
    print(f"  {G.nodes[nid].content!r:15s} {amp:7.2f}")

БЕЗ вентиля:
  'beverage'       146.67
  'coffee'         100.00
  'hot drink'      100.00
  'tea'            100.00
  'caffeine'        26.67
  'juice'           23.47
  'soda'            23.47


In [7]:
# С вентилем — те же vecs_2, что дали якоря, теперь ещё и гейтируют диффузию
lit_gated = diffuse(G, source_amplitudes_2, decay=0.8, min_propagate_fraction=0.01, max_steps=20,
                     query_vectors=vecs_2)

print("С вентилем rel(B|q):")
for nid, amp in sorted(lit_gated.items(), key=lambda kv: -kv[1]):
    marker = " <-- ожидаемый ответ" if G.nodes[nid].content == "caffeine" else ""
    marker = " <-- дистрактор" if G.nodes[nid].content == "beverage" else marker
    print(f"  {G.nodes[nid].content!r:15s} {amp:7.2f}{marker}")

caffeine_id = next(n.id for n in G.nodes.values() if n.content == "caffeine")
beverage_id = next(n.id for n in G.nodes.values() if n.content == "beverage")
print()
print(f"caffeine \u0432\u044b\u0438\u0433\u0440\u0430\u043b beverage:", lit_gated[caffeine_id] > lit_gated[beverage_id])

С вентилем rel(B|q):
  'beverage'       129.46 <-- дистрактор
  'coffee'         100.00
  'hot drink'      100.00
  'tea'            100.00
  'caffeine'        27.20 <-- ожидаемый ответ
  'soda'            13.65
  'juice'           11.49

caffeine выиграл beverage: False


## Регресс: multi-word (north dakota) и мульти-якорный запрос всё ещё работают с вентилем

In [8]:
QUERY_1 = "tell me about north dakota"
vecs_1 = query_vectors(QUERY_1, numberbatch)
anchors_1 = set()
for v in vecs_1:
    anchors_1.update(nearest_nodes_by_embedding(G, v, top_k=3))
lit_1 = diffuse(G, {a: 100.0 for a in anchors_1}, decay=0.8, min_propagate_fraction=0.01,
                max_steps=20, query_vectors=vecs_1)
print("north dakota query, с вентилем, топ-5:")
for nid, amp in sorted(lit_1.items(), key=lambda kv: -kv[1])[:5]:
    print(f"  {G.nodes[nid].content!r:15s} {amp:7.2f}")

north dakota query, с вентилем, топ-5:
  'south dakota'   100.00
  'north dakota'   100.00
